In [1]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: C:\Users\User\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd

df = pd.DataFrame({
    "id": pd.Series(dtype="int64"),
    "name": pd.Series(dtype="str"),
    "age": pd.Series(dtype="int64"),
    "salary": pd.Series(dtype="float64"),
    "department": pd.Series(dtype="str"),
    "hiredate": pd.Series(dtype="datetime64[ns]"),
    "is_active": pd.Series(dtype="bool")
})

print(df)


Empty DataFrame
Columns: [id, name, age, salary, department, hiredate, is_active]
Index: []


In [3]:
import pandas as pd
import os

df = pd.DataFrame({
    "id": pd.Series(dtype="int64"),
    "name": pd.Series(dtype="str"),
    "age": pd.Series(dtype="int64"),
    "salary": pd.Series(dtype="float64"),
    "department": pd.Series(dtype="str"),
    "hiredate": pd.Series(dtype="datetime64[ns]"),
    "is_active": pd.Series(dtype="bool")
})


df.to_csv("employees_with_index.csv")                
df.to_csv("employees_no_index.csv", index=False)     

size_with = os.path.getsize("employees_with_index.csv")
size_no = os.path.getsize("employees_no_index.csv")

print(f"Размер с индексом: {size_with} байт")
print(f"Размер без индекса: {size_no} байт")
print(f"Разница: {size_with - size_no} байт\n")

df_default = pd.read_csv("employees_no_index.csv")
print("Типы при обычном чтении")
print(df_default.dtypes)

df_fast = pd.read_csv(
    "employees_no_index.csv",
    usecols=["id", "name", "salary", "department", "hiredate", "is_active"],
    dtype={
        "id": "int32",             
        "salary": "float32",      
        "department": "category",  
        "is_active": "bool"
    },
    parse_dates=["hiredate"]       
)

print("\nТипы при оптимизированном чтении")
print(df_fast.dtypes)

Размер с индексом: 51 байт
Размер без индекса: 50 байт
Разница: 1 байт

Типы при обычном чтении
id            object
name          object
age           object
salary        object
department    object
hiredate      object
is_active     object
dtype: object

Типы при оптимизированном чтении
id               int32
name            object
salary         float32
department    category
hiredate        object
is_active         bool
dtype: object


In [4]:
import os
import json
import pandas as pd

df = pd.DataFrame({
    "id":[1, 2, 3],
    "name": ["Алексей", "Мария", "Иван"],
    "skills": [
        ["Python", "SQL", "Pandas"],
        ["Figma", "Marketing"],
        ["Python", "Spark", "Airflow", "Docker"]
    ],
    "contacts": [
        {"city": "Москва", "remote": True},
        {"city": "Тула", "remote": False},
        {"city": "Казань", "remote": True}
    ]
})


orientations = ["records", "columns", "index", "split"]

print("Размеры файлов")
for orient in orientations:
    filename = f"employees_{orient}.json"
    df.to_json(filename, orient=orient, force_ascii=False, indent=2)
    size = os.path.getsize(filename)
    print(f"orient='{orient}': {size} байт")

df_loaded = pd.read_json("employees_records.json", orient="records")
print("\nПрочитанный DataFrame")
print(df_loaded)

with open("employees_records.json", "r", encoding="utf-8") as f:
    raw_data = json.load(f)

df_flat_dicts = pd.json_normalize(raw_data)
print("\nРазвёрнутые словари (contacts)")
print(df_flat_dicts)

df_flat_skills = pd.json_normalize(
    raw_data,
    record_path=["skills"],
    meta=["id", "name", ["contacts", "city"]]
).rename(columns={0: "skill"})

print("\nРазвёрнутый список навыков (record_path)")
print(df_flat_skills)


Размеры файлов
orient='records': 561 байт
orient='columns': 553 байт
orient='index': 573 байт
orient='split': 665 байт

Прочитанный DataFrame
   id     name                            skills  \
0   1  Алексей             [Python, SQL, Pandas]   
1   2    Мария                [Figma, Marketing]   
2   3     Иван  [Python, Spark, Airflow, Docker]   

                             contacts  
0  {'city': 'Москва', 'remote': True}  
1   {'city': 'Тула', 'remote': False}  
2  {'city': 'Казань', 'remote': True}  

Развёрнутые словари (contacts)
   id     name                            skills contacts.city  \
0   1  Алексей             [Python, SQL, Pandas]        Москва   
1   2    Мария                [Figma, Marketing]          Тула   
2   3     Иван  [Python, Spark, Airflow, Docker]        Казань   

   contacts.remote  
0             True  
1            False  
2             True  

Развёрнутый список навыков (record_path)
       skill id     name contacts.city
0     Python  1  Алексей   

In [5]:
pip install pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: C:\Users\User\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [6]:
import os
import time
import numpy as np
import pandas as pd

n_rows = 100000
df = pd.DataFrame({
    "id": np.arange(1, n_rows + 1),
    "name": np.random.choice(["Алексей", "Мария", "Иван", "Елена"], n_rows),
    "age": np.random.randint(22, 60, n_rows),
    "salary": np.round(np.random.uniform(70000, 250000, n_rows), 2),
    "department": np.random.choice(["IT", "Маркетинг", "Аналитика", "HR"], n_rows),
    "hiredate": pd.to_datetime("2020-01-01") + pd.to_timedelta(np.random.randint(0, 1500, n_rows), unit="D"),
    "is_active": np.random.choice([True, False], n_rows)
})

df.to_csv("data.csv", index=False)

compressions = ["snappy", "gzip", "brotli"]
for comp in compressions:
    df.to_parquet(f"data_{comp}.parquet", engine="pyarrow", compression=comp)

print("Размеры файлов (100000 строк)")
files = ["data.csv"] + [f"data_{c}.parquet" for c in compressions]
for f in files:
    size_kb = os.path.getsize(f) / 1024
    print(f"{f:<22}: {size_kb:>8.2f} КБ")

t0 = time.perf_counter()
df_csv = pd.read_csv("data.csv")
t_csv = time.perf_counter() - t0

t0 = time.perf_counter()
df_pq = pd.read_parquet("data_snappy.parquet", engine="pyarrow")
t_pq = time.perf_counter() - t0

print(f"\nВремя полного чтения")
print(f"CSV:              {t_csv:.4f} сек")
print(f"Parquet (snappy): {t_pq:.4f} сек")

t0 = time.perf_counter()
df_pruned = pd.read_parquet("data_snappy.parquet", columns=["id", "salary"], engine="pyarrow")
t_pruned = time.perf_counter() - t0

print(f"\nColumn Pruning (чтение только 2 столбцов из 7)")
print(f"Время чтения:     {t_pruned:.4f} сек")
print(f"Загруженные столбцы: {list(df_pruned.columns)}")

print("\nСравнение типов данных после чтения")
comparison = pd.DataFrame({
    "Исходный DF": df.dtypes,
    "После CSV": df_csv.dtypes,
    "После Parquet": df_pq.dtypes
})
print(comparison)


Размеры файлов (100000 строк)
data.csv              :  5722.85 КБ
data_snappy.parquet   :  1633.42 КБ
data_gzip.parquet     :  1200.41 КБ
data_brotli.parquet   :  1096.75 КБ

Время полного чтения
CSV:              0.0310 сек
Parquet (snappy): 0.0680 сек

Column Pruning (чтение только 2 столбцов из 7)
Время чтения:     0.0080 сек
Загруженные столбцы: ['id', 'salary']

Сравнение типов данных после чтения
               Исходный DF После CSV   После Parquet
id                   int64     int64           int64
name                   str       str             str
age                  int32     int64           int32
salary             float64   float64         float64
department             str       str             str
hiredate    datetime64[us]       str  datetime64[us]
is_active             bool      bool            bool


In [7]:
pip install tables

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: C:\Users\User\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [8]:
import os
import numpy as np
import pandas as pd

n_rows = 100000
df = pd.DataFrame({
    "id": np.arange(1, n_rows + 1),
    "name": np.random.choice(["Алексей", "Мария", "Иван", "Елена"], n_rows),
    "age": np.random.randint(22, 60, n_rows),
    "salary": np.round(np.random.uniform(70000, 250000, n_rows), 2),
    "department": np.random.choice(["IT", "Маркетинг", "Аналитика", "HR"], n_rows),
    "hiredate": pd.to_datetime("2020-01-01") + pd.to_timedelta(np.random.randint(0, 1500, n_rows), unit="D"),
    "is_active": np.random.choice([True, False], n_rows)
})

df_agg = df.groupby("department", as_index=False).agg(
    emp_count=("id", "count"),
    avg_salary=("salary", "mean"),
    max_age=("age", "max")
)

hdf_file = "company_data.h5"
if os.path.exists(hdf_file):
    os.remove(hdf_file)

df.to_hdf(hdf_file, key="employees_fixed", mode="a", format="fixed")
df_agg.to_hdf(hdf_file, key="departments_summary", mode="a", format="fixed")

df.to_hdf(
    hdf_file,
    key="employees_table",
    mode="a",
    format="table",
    data_columns=["department", "salary", "is_active"]
)

with pd.HDFStore(hdf_file, mode="r") as store:
    print("Ключи внутри HDF5:", store.keys())

df_summary_loaded = pd.read_hdf(hdf_file, key="departments_summary")
print("\nПрочитанная таблица по ключу 'departments_summary'")
print(df_summary_loaded)

df_filtered = pd.read_hdf(
    hdf_file,
    key="employees_table",
    where="department == 'IT' & salary > 200000 & is_active == True"
)

print(f"\nРезультат запроса с диска (найдено строк: {len(df_filtered)} из {len(df)}) ")
print(df_filtered.head())

df.to_csv("employees.csv", index=False)
df.to_hdf("only_fixed.h5", key="data", mode="w", format="fixed")
df.to_hdf("only_table.h5", key="data", mode="w", format="table", data_columns=["department", "salary", "is_active"])

print("\nСравнение размеров файлов (100 000 строк)")
for fname in ["employees.csv", "only_fixed.h5", "only_table.h5", hdf_file]:
    size_mb = os.path.getsize(fname) / (1024 * 1024)
    print(f"{fname:<20}: {size_mb:.2f} МБ")


Ключи внутри HDF5: ['/departments_summary', '/employees_fixed', '/employees_table']

Прочитанная таблица по ключу 'departments_summary'
  department  emp_count     avg_salary  max_age
0         HR      25257  159731.296482       59
1         IT      24815  159889.127688       59
2  Аналитика      24961  160462.322953       59
3  Маркетинг      24967  159847.652087       59

Результат запроса с диска (найдено строк: 3436 из 100000) 
      id     name  age     salary department   hiredate  is_active
16    17     Иван   46  238742.29         IT 2023-10-03       True
67    68  Алексей   51  204585.90         IT 2021-12-10       True
170  171  Алексей   48  245431.94         IT 2022-01-28       True
171  172  Алексей   42  222349.63         IT 2021-03-07       True
194  195    Мария   47  230801.97         IT 2020-08-08       True

Сравнение размеров файлов (100 000 строк)
employees.csv       : 5.59 МБ
only_fixed.h5       : 8.08 МБ
only_table.h5       : 7.32 МБ
company_data.h5     : 16.41 М

In [9]:
import os
import time
import numpy as np
import pandas as pd

n_rows = 100000
df = pd.DataFrame({
    "id": np.arange(1, n_rows + 1),
    "name": np.random.choice(["Алексей", "Мария", "Иван", "Елена"], n_rows),
    "age": np.random.randint(22, 60, n_rows),
    "salary": np.round(np.random.uniform(70000, 250000, n_rows), 2),
    "department": np.random.choice(["IT", "Маркетинг", "Аналитика", "HR"], n_rows),
    "hiredate": pd.to_datetime("2020-01-01") + pd.to_timedelta(np.random.randint(0, 1500, n_rows), unit="D"),
    "is_active": np.random.choice([True, False], n_rows)
})

benchmarks = [
    (
        "CSV (обычный)",
        "bench.csv",
        lambda f: df.to_csv(f, index=False),
        lambda f: pd.read_csv(f, parse_dates=["hiredate"])
    ),
    (
        "JSON (records)",
        "bench.json",
        lambda f: df.to_json(f, orient="records", force_ascii=False),
        lambda f: pd.read_json(f, orient="records")
    ),
    (
        "Parquet (snappy)",
        "bench_snappy.parquet",
        lambda f: df.to_parquet(f, engine="pyarrow", compression="snappy"),
        lambda f: pd.read_parquet(f, engine="pyarrow")
    ),
    (
        "Parquet (brotli)",
        "bench_brotli.parquet",
        lambda f: df.to_parquet(f, engine="pyarrow", compression="brotli"),
        lambda f: pd.read_parquet(f, engine="pyarrow")
    ),
    (
        "HDF5 (fixed, без сжатия)",
        "bench_fixed.h5",
        lambda f: df.to_hdf(f, key="data", mode="w", format="fixed"),
        lambda f: pd.read_hdf(f, key="data")
    ),
    (
        "HDF5 (table + blosc сжатие)",
        "bench_table.h5",
        lambda f: df.to_hdf(f, key="data", mode="w", format="table", complib="blosc", complevel=5),
        lambda f: pd.read_hdf(f, key="data")
    )
]

results = []

for name, filename, write_fn, read_fn in benchmarks:
    # Замер времени записи
    t0 = time.perf_counter()
    write_fn(filename)
    write_time = time.perf_counter() - t0

    # Замер размера файла
    size_mb = os.path.getsize(filename) / (1024 * 1024)

    # Замер времени чтения
    t0 = time.perf_counter()
    _ = read_fn(filename)
    read_time = time.perf_counter() - t0

    results.append({
        "Формат": name,
        "Размер (МБ)": round(size_mb, 2),
        "Запись (сек)": round(write_time, 4),
        "Чтение (сек)": round(read_time, 4)
    })

    if os.path.exists(filename):
        os.remove(filename)

report_df = pd.DataFrame(results)
csv_size = report_df.loc[0, "Размер (МБ)"]
csv_read = report_df.loc[0, "Чтение (сек)"]

report_df["Сжатие к CSV"] = (csv_size / report_df["Размер (МБ)"]).round(1).astype(str) + "x"
report_df["Ускорение чтения к CSV"] = (csv_read / report_df["Чтение (сек)"]).round(1).astype(str) + "x"

print("ИТОГОВЫЙ ОТЧЁТ ПО ФОРМАТАМ (100 000 строк)")
print(report_df.to_string(index=False))

# Для аналитики: Parquet
# Для обмена: CSV и JSON
# Для научных расчётов HDF5

C:\Users\User\AppData\Local\Temp\ipykernel_3960\2560449166.py:27: Pandas4Warning: The default formatting of datetime/timedelta values will change from numbers ("epoch") to strings ("iso") in a future version. Specify `date_format="iso"` explicitly in the `to_json()` call to opt-in to the future behaviour and silence this warning.
  lambda f: df.to_json(f, orient="records", force_ascii=False),


ИТОГОВЫЙ ОТЧЁТ ПО ФОРМАТАМ (100 000 строк)
                     Формат  Размер (МБ)  Запись (сек)  Чтение (сек) Сжатие к CSV Ускорение чтения к CSV
              CSV (обычный)         5.60        0.4810        0.0566         1.0x                   1.0x
             JSON (records)        12.37        0.2507        0.5853         0.5x                   0.1x
           Parquet (snappy)         1.59        0.0648        0.0168         3.5x                   3.4x
           Parquet (brotli)         1.07        0.4172        0.0202         5.2x                   2.8x
   HDF5 (fixed, без сжатия)         8.08        0.1094        0.1253         0.7x                   0.5x
HDF5 (table + blosc сжатие)         3.10        0.2492        0.2306         1.8x                   0.2x


In [10]:
import pandas as pd

orders = pd.read_csv("orders.csv")

df1 = pd.DataFrame({
    "client_clean": orders["client"].astype(str).str.strip().str.title(),
    "city_clean": orders["city"].astype(str).str.strip().str.title(),
    "category_clean": orders["category"].astype(str).str.strip().str.lower(),
    "product_clean": orders["product"].astype(str).str.strip().str.upper(),
    "revenue": (orders["qty"] * orders["price"] * (1 - orders["discount"])).round(2)
})

print(df1.head())


     client_clean        city_clean category_clean product_clean   revenue
0     Ivan Petrov            Moscow    electronics        LAPTOP  135000.0
1     Ivan Petrov            Moscow    electronics        LAPTOP   68400.0
2   Anna Smirnova  Saint Petersburg           home         CHAIR   13500.0
3    Pavel Ivanov             Kazan    electronics       MONITOR   15300.0
4  Maria Sokolova       Novosibirsk         office          DESK   19200.0


In [11]:
import pandas as pd

orders = pd.read_csv("orders.csv", parse_dates=["order_date"])

df1 = pd.DataFrame({
    "order_date": orders["order_date"],
    "client_clean": orders["client"].astype(str).str.strip().str.title(),
    "city_clean": orders["city"].astype(str).str.strip().str.title(),
    "category_clean": orders["category"].astype(str).str.strip().str.lower(),
    "product_clean": orders["product"].astype(str).str.strip().str.upper(),
    "revenue": (orders["qty"] * orders["price"] * (1 - orders["discount"])).round(2)
})

df2 = (
    df1
    .assign(
        revenue_rank=df1["revenue"].rank(method="dense", ascending=False).astype(int)
    )
    .sort_values(
        by=["revenue", "order_date"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

print(df2)


   order_date        client_clean        city_clean category_clean  \
0  2024-01-05         Ivan Petrov            Moscow    electronics   
1  2024-02-14       Anna Smirnova  Saint Petersburg    electronics   
2  2024-01-05         Ivan Petrov            Moscow    electronics   
3  2024-01-22     Olga Kuznetsova             Kazan    electronics   
4  2024-02-22        Alexey Popov     Yekaterinburg    electronics   
5  2024-02-14       Anna Smirnova  Saint Petersburg    electronics   
6  2024-04-03       Viktor Osipov             Kazan           home   
7  2024-01-18       Elena Petrova            Moscow           home   
8  2024-03-10       Maxim Semenov             Kazan    electronics   
9  2024-03-22       Galina Belova             Kazan         office   
10 2024-03-05       Andrey Egorov       Novosibirsk           home   
11 2024-03-20        Denis Frolov     Yekaterinburg           home   
12 2024-03-15      Roman Bogdanov  Saint Petersburg         office   
13 2024-01-12      M

In [12]:
import pandas as pd

orders = pd.read_csv("orders.csv")

df1 = pd.DataFrame({
    "order_date": orders["order_date"],
    "delivery_days": orders["delivery_days"],
    "client_clean": orders["client"].astype(str).str.strip().str.title(),
    "city_clean": orders["city"].astype(str).str.strip().str.title(),
    "category_clean": orders["category"].astype(str).str.strip().str.lower(),
    "product_clean": orders["product"].astype(str).str.strip().str.upper(),
    "revenue": (orders["qty"] * orders["price"] * (1 - orders["discount"])).round(2)
})

df2 = (
    df1
    .assign(
        revenue_rank=df1["revenue"].rank(method="dense", ascending=False).astype(int)
    )
    .sort_values(by=["revenue", "order_date"], ascending=[False, True])
    .reset_index(drop=True)
)

df3 = df2.copy()

df3["order_date"] = pd.to_datetime(df3["order_date"])

df3["order_year"] = df3["order_date"].dt.year
df3["order_month"] = df3["order_date"].dt.month
df3["order_quarter"] = df3["order_date"].dt.quarter
df3["order_weekday"] = df3["order_date"].dt.day_name()
df3["is_weekend"] = df3["order_date"].dt.dayofweek.isin([5, 6])
df3["delivery_date"] = df3["order_date"] + pd.to_timedelta(df3["delivery_days"], unit="D")
df3["days_to_month_end"] = df3["order_date"].dt.days_in_month - df3["order_date"].dt.day

print(df3)


   order_date  delivery_days        client_clean        city_clean  \
0  2024-01-05              3         Ivan Petrov            Moscow   
1  2024-02-14              4       Anna Smirnova  Saint Petersburg   
2  2024-01-05              5         Ivan Petrov            Moscow   
3  2024-01-22              3     Olga Kuznetsova             Kazan   
4  2024-02-22              3        Alexey Popov     Yekaterinburg   
5  2024-02-14              2       Anna Smirnova  Saint Petersburg   
6  2024-04-03              7       Viktor Osipov             Kazan   
7  2024-01-18             10       Elena Petrova            Moscow   
8  2024-03-10              3       Maxim Semenov             Kazan   
9  2024-03-22              2       Galina Belova             Kazan   
10 2024-03-05              0       Andrey Egorov       Novosibirsk   
11 2024-03-20              6        Denis Frolov     Yekaterinburg   
12 2024-03-15              5      Roman Bogdanov  Saint Petersburg   
13 2024-01-12       

C:\Users\User\AppData\Local\Temp\ipykernel_3960\3373413136.py:32: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df3["is_weekend"] = df3["order_date"].dt.dayofweek.isin([5, 6])


In [13]:
import pandas as pd

orders = pd.read_csv("orders.csv")

df1 = pd.DataFrame({
    "order_id": orders["order_id"],
    "order_date": orders["order_date"],
    "delivery_days": orders["delivery_days"],
    "client_clean": orders["client"].astype(str).str.strip().str.title(),
    "city_clean": orders["city"].astype(str).str.strip().str.title(),
    "category_clean": orders["category"].astype(str).str.strip().str.lower(),
    "product_clean": orders["product"].astype(str).str.strip().str.upper(),
    "revenue": (orders["qty"] * orders["price"] * (1 - orders["discount"])).round(2)
})

df2 = (
    df1
    .assign(revenue_rank=df1["revenue"].rank(method="dense", ascending=False).astype(int))
    .sort_values(by=["revenue", "order_date"], ascending=[False, True])
    .reset_index(drop=True)
)

df3 = df2.copy()
df3["order_date"] = pd.to_datetime(df3["order_date"])
df3["order_year"] = df3["order_date"].dt.year
df3["order_month"] = df3["order_date"].dt.month
df3["order_quarter"] = df3["order_date"].dt.quarter
df3["order_weekday"] = df3["order_date"].dt.day_name()
df3["is_weekend"] = df3["order_date"].dt.dayofweek >= 5
df3["delivery_date"] = df3["order_date"] + pd.to_timedelta(df3["delivery_days"], unit="D")
df3["days_to_month_end"] = df3["order_date"].dt.days_in_month - df3["order_date"].dt.day

dup_cols = ["client_clean", "order_date", "product_clean"]

df4 = (
    df3
    .assign(duplicate_group_flag=df3.duplicated(subset=dup_cols, keep=False))
    .sort_values(by="revenue", ascending=False)
    .drop_duplicates(subset=dup_cols, keep="first")
    .sort_values(by="order_id", ascending=True)
    .reset_index(drop=True)
)

print(df4)


    order_id order_date  delivery_days        client_clean        city_clean  \
0          1 2024-01-05              3         Ivan Petrov            Moscow   
1          3 2024-01-07              7       Anna Smirnova  Saint Petersburg   
2          4 2024-01-10              4        Pavel Ivanov             Kazan   
3          5 2024-01-12              6      Maria Sokolova       Novosibirsk   
4          6 2024-01-15              0       Sergey Volkov     Yekaterinburg   
5          7 2024-01-18             10       Elena Petrova            Moscow   
6          8 2024-01-20              8      Dmitry Smirnov  Saint Petersburg   
7          9 2024-01-22              3     Olga Kuznetsova             Kazan   
8         11 2024-02-14              4       Anna Smirnova  Saint Petersburg   
9         12 2024-02-16              5        Igor Fedorov            Moscow   
10        13 2024-02-20              2    Natalia Morozova       Novosibirsk   
11        14 2024-02-22              3  

C:\Users\User\AppData\Local\Temp\ipykernel_3960\2070325696.py:29: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df3["is_weekend"] = df3["order_date"].dt.dayofweek >= 5


In [14]:
import numpy as np
import pandas as pd

orders = pd.read_csv("orders.csv")

df1 = pd.DataFrame({
    "order_id": orders["order_id"],
    "order_date": orders["order_date"],
    "delivery_days": orders["delivery_days"],
    "status": orders["status"],
    "promo_code": orders["promo_code"],
    "client_clean": orders["client"].astype(str).str.strip().str.title(),
    "city_clean": orders["city"].astype(str).str.strip().str.title(),
    "category_clean": orders["category"].astype(str).str.strip().str.lower(),
    "product_clean": orders["product"].astype(str).str.strip().str.upper(),
    "revenue": (orders["qty"] * orders["price"] * (1 - orders["discount"])).round(2)
})

df2 = (
    df1
    .assign(revenue_rank=df1["revenue"].rank(method="dense", ascending=False).astype(int))
    .sort_values(by=["revenue", "order_date"], ascending=[False, True])
    .reset_index(drop=True)
)

df3 = df2.copy()
df3["order_date"] = pd.to_datetime(df3["order_date"])
df3["order_year"] = df3["order_date"].dt.year
df3["order_month"] = df3["order_date"].dt.month
df3["order_quarter"] = df3["order_date"].dt.quarter
df3["order_weekday"] = df3["order_date"].dt.day_name()
df3["is_weekend"] = df3["order_date"].dt.dayofweek >= 5
df3["delivery_date"] = df3["order_date"] + pd.to_timedelta(df3["delivery_days"], unit="D")
df3["days_to_month_end"] = df3["order_date"].dt.days_in_month - df3["order_date"].dt.day

dup_cols = ["client_clean", "order_date", "product_clean"]
df4 = (
    df3
    .assign(duplicate_group_flag=df3.duplicated(subset=dup_cols, keep=False))
    .sort_values(by="revenue", ascending=False)
    .drop_duplicates(subset=dup_cols, keep="first")
    .sort_values(by="order_id", ascending=True)
    .reset_index(drop=True)
)

df5 = df4.copy()

df5["revenue_segment"] = pd.cut(
    df5["revenue"],
    bins=[0, 1000, 5000, 20000, np.inf],
    labels=["low", "medium", "high", "vip"],
    right=False
)

status_map = {
    "new": "новый",
    "paid": "оплачен",
    "shipped": "отправлен",
    "cancelled": "отменён",
    "returned": "возврат"
}
df5["status_ru"] = df5["status"].astype(str).str.strip().str.lower().map(status_map)

promo_clean = df5["promo_code"].fillna("NO_PROMO").astype(str).str.strip()
df5["promo_used"] = promo_clean != "NO_PROMO"

print(df5)


C:\Users\User\AppData\Local\Temp\ipykernel_3960\3892238242.py:32: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df3["is_weekend"] = df3["order_date"].dt.dayofweek >= 5


    order_id order_date  delivery_days     status promo_code  \
0          1 2024-01-05              3       paid     SAVE10   
1          3 2024-01-07              7        new    WELCOME   
2          4 2024-01-10              4       paid     SAVE10   
3          5 2024-01-12              6    shipped   NO_PROMO   
4          6 2024-01-15              0  cancelled   NO_PROMO   
5          7 2024-01-18             10       paid      VIP25   
6          8 2024-01-20              8   returned   NO_PROMO   
7          9 2024-01-22              3    shipped     SAVE10   
8         11 2024-02-14              4        new   NO_PROMO   
9         12 2024-02-16              5    shipped   NO_PROMO   
10        13 2024-02-20              2       paid     SAVE10   
11        14 2024-02-22              3       paid      VIP25   
12        15 2024-02-25              9    shipped   NO_PROMO   
13        16 2024-03-01              6        new    WELCOME   
14        17 2024-03-03              4  

In [15]:
import numpy as np
import pandas as pd

orders = pd.read_csv("orders.csv")

df1 = pd.DataFrame({
    "order_id": orders["order_id"],
    "order_date": orders["order_date"],
    "delivery_days": orders["delivery_days"],
    "status": orders["status"],
    "promo_code": orders["promo_code"],
    "manager": orders["manager"],
    "client_clean": orders["client"].astype(str).str.strip().str.title(),
    "city_clean": orders["city"].astype(str).str.strip().str.title(),
    "category_clean": orders["category"].astype(str).str.strip().str.lower(),
    "product_clean": orders["product"].astype(str).str.strip().str.upper(),
    "revenue": (orders["qty"] * orders["price"] * (1 - orders["discount"])).round(2)
})

df2 = (
    df1
    .assign(revenue_rank=df1["revenue"].rank(method="dense", ascending=False).astype(int))
    .sort_values(by=["revenue", "order_date"], ascending=[False, True])
    .reset_index(drop=True)
)

df3 = df2.copy()
df3["order_date"] = pd.to_datetime(df3["order_date"])
df3["order_year"] = df3["order_date"].dt.year
df3["order_month"] = df3["order_date"].dt.month
df3["order_quarter"] = df3["order_date"].dt.quarter
df3["order_weekday"] = df3["order_date"].dt.day_name()
df3["is_weekend"] = df3["order_date"].dt.dayofweek >= 5
df3["delivery_date"] = df3["order_date"] + pd.to_timedelta(df3["delivery_days"], unit="D")
df3["days_to_month_end"] = df3["order_date"].dt.days_in_month - df3["order_date"].dt.day

dup_cols = ["client_clean", "order_date", "product_clean"]
df4 = (
    df3
    .assign(duplicate_group_flag=df3.duplicated(subset=dup_cols, keep=False))
    .sort_values(by="revenue", ascending=False)
    .drop_duplicates(subset=dup_cols, keep="first")
    .sort_values(by="order_id", ascending=True)
    .reset_index(drop=True)
)

df5 = df4.copy()
df5["revenue_segment"] = pd.cut(
    df5["revenue"],
    bins=[0, 1000, 5000, 20000, np.inf],
    labels=["low", "medium", "high", "vip"],
    right=False
)
status_map = {
    "new": "новый",
    "paid": "оплачен",
    "shipped": "отправлен",
    "cancelled": "отменён",
    "returned": "возврат"
}
df5["status_ru"] = df5["status"].astype(str).str.strip().str.lower().map(status_map)
promo_clean = df5["promo_code"].fillna("NO_PROMO").astype(str).str.strip()
df5["promo_used"] = promo_clean != "NO_PROMO"

df6 = df5.copy()

status_clean = df6["status"].astype(str).str.strip().str.lower()

conditions = [
    status_clean.isin(["cancelled", "returned"]),
    (df6["revenue"] >= 20000) & (df6["delivery_days"] <= 3),
    df6["revenue"] >= 5000
]
choices = ["archive", "A", "B"]

df6["priority"] = np.select(conditions, choices, default="C")

manager_clean = (
    df6["manager"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", ".", regex=True)
)
df6["manager_email"] = manager_clean + "@shop.ru"

df6["final_comment"] = (
    df6["client_clean"]
    + ": "
    + df6["priority"]
    + ", "
    + df6["revenue_segment"].astype(str)
)

df6.to_csv("orders_final.csv", index=False, encoding="utf-8-sig")

print(df6[["order_id", "client_clean", "priority", "manager_email", "final_comment"]])


C:\Users\User\AppData\Local\Temp\ipykernel_3960\1674800314.py:33: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df3["is_weekend"] = df3["order_date"].dt.dayofweek >= 5


    order_id        client_clean priority           manager_email  \
0          1         Ivan Petrov        A    ivan.sidorov@shop.ru   
1          3       Anna Smirnova        B  petr.kuznetsov@shop.ru   
2          4        Pavel Ivanov        B    anna.volkova@shop.ru   
3          5      Maria Sokolova        B   olga.morozova@shop.ru   
4          6       Sergey Volkov  archive    dmitry.orlov@shop.ru   
5          7       Elena Petrova        B    ivan.sidorov@shop.ru   
6          8      Dmitry Smirnov  archive  petr.kuznetsov@shop.ru   
7          9     Olga Kuznetsova        A    anna.volkova@shop.ru   
8         11       Anna Smirnova        B  petr.kuznetsov@shop.ru   
9         12        Igor Fedorov        B    ivan.sidorov@shop.ru   
10        13    Natalia Morozova        C   olga.morozova@shop.ru   
11        14        Alexey Popov        A    dmitry.orlov@shop.ru   
12        15    Tatiana Lebedeva        B    anna.volkova@shop.ru   
13        16     Vladimir Kozlov  